# Урок 5. Multiple Linear Regression — много признаков
**Библиотеки:** numpy, pandas, sklearn

**Цель:** предсказывать по нескольким признакам сразу и понять векторизацию.

## Простыми словами
В жизни цена квартиры зависит не только от площади: ещё комнаты, этаж, возраст дома...
Модель становится: `f(x) = w1*x1 + w2*x2 + ... + wn*xn + b` — у КАЖДОГО признака свой вес.

**Векторизация:** вместо суммы по одному — одно матричное умножение `X @ w + b`.
Символ @ = скалярное произведение/умножение матриц. NumPy делает это молниеносно.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
n = 100
df = pd.DataFrame({
    "площадь":   rng.uniform(30, 150, n).round(0),
    "комнаты":   rng.integers(1, 5, n),
    "этаж":      rng.integers(1, 16, n),
    "возраст":   rng.integers(0, 40, n),
})
# "Настоящая" формула цены + шум (в жизни формулу никто не знает — её ищет модель)
df["цена"] = (2.5*df["площадь"] + 15*df["комнаты"] - 0.8*df["возраст"]
              + 50 + rng.normal(0, 12, n)).round(1)

X = df[["площадь", "комнаты", "этаж", "возраст"]]
y = df["цена"]
print(df.head())
print("X shape:", X.shape, "| y shape:", y.shape)   # (100, 4) и (100,)

model = LinearRegression().fit(X, y)
print("\nВеса признаков:")
for name, w in zip(X.columns, model.coef_):
    print(f"  {name}: {w:+.2f}")
print("b:", round(model.intercept_, 2))

In [ ]:
# Предсказание для новой квартиры: 75 м2, 2 комнаты, 5 этаж, 10 лет
new_apt = pd.DataFrame([[75, 2, 5, 10]], columns=X.columns)
print("Цена:", round(model.predict(new_apt)[0], 1), "тыс. $")

# Векторизация вручную: то же предсказание через X @ w + b
w_vec = model.coef_
manual = new_apt.values @ w_vec + model.intercept_
print("Вручную через @:", round(manual[0], 1))   # совпадает!

## Что произошло
Модель нашла веса, близкие к «настоящим» (2.5, 15, ~0, −0.8): площадь сильно влияет «+»,
возраст «−», этаж почти не влияет (мы его и не закладывали в формулу — модель это поняла!).
Предсказание = строка признаков @ вектор весов + b. Одна операция вместо цикла.

## Типичные ошибки
1. Сравнивать веса признаков РАЗНЫХ масштабов: «комнаты важнее площади, ведь 15 > 2.5» —
   нельзя так судить без масштабирования (урок 6).
2. predict с перепутанным порядком колонок — веса применятся не к тем признакам.
3. Забыть, что shape X теперь (n, 4): 4 колонки = 4 веса.

## Конспект 📓
Много признаков: f(x) = w1x1+...+wnxn+b, по весу на признак. coef_ — массив весов.
Векторизация: X @ w + b. Порядок колонок при predict — тот же, что при fit.

---
## Мини-задание 💪
Добавь в датасет признак «расстояние до центра» (влияет на цену с минусом), пересоздай y, обучи заново. Нашла ли модель отрицательный вес?

## 5 проверочных вопросов ❓
1. Как выглядит формула модели с 4 признаками?
2. Что хранится в model.coef_ при нескольких признаках?
3. Что делает операция X @ w?
4. Почему нельзя сравнивать «важность» весов без масштабирования?
5. Какой shape у X для 200 квартир и 6 признаков?

## Домашнее задание 🏠
Задача 41 из practice_tasks.md (векторный gradient descent). Это сложно — используй промпт 18, если застрянешь.

> Не понял что-то? Открой prompts_for_future.md — промпты 1–6 помогут разобрать любую тему с AI.
